<a href="https://colab.research.google.com/github/AyseNil-star/AyseNil-star/blob/main/week03_comparing_interfaces_ipynb_json.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 3 — Comparing Interfaces

**Computational Antibody Engineering**

You can now measure one interface (week 1) and name any residue unambiguously (week 2). Today you
do the thing those two weeks were building toward: **compare two antibodies against the same antigen.**

The pair is real and clinically important. **Trastuzumab** and **pertuzumab** both target HER2. Both
are approved. They are routinely given *together*, and the reason is structural — you are about to
compute it.

Today's questions:

> **Do trastuzumab and pertuzumab compete for the same surface of HER2?**
> **And is "the interface" even a stable thing to measure?**

The second question has an uncomfortable answer, and we get it almost for free.

---
### What you already have
Everything today reuses week 1's contact calculation and week 2's numbering. You will write both as
reusable functions this time, because from here on you will keep needing them.

## 0. Setup

Same stack as week 2 — ANARCI needs HMMER. Start this now; it takes 1–2 minutes.

In [ ]:
!apt-get install -qq hmmer > /dev/null 2>&1
%pip install -q anarci biopython pandas matplotlib

import warnings
warnings.filterwarnings("ignore")

import shutil
assert shutil.which("hmmscan"), "hmmscan not found - re-run this cell"

from Bio.PDB import PDBParser, NeighborSearch
from Bio.PDB.Polypeptide import three_to_index, index_to_one
from Bio.PDB.SASA import ShrakeRupley
from anarci import number
from collections import Counter
import pandas as pd, matplotlib.pyplot as plt
import urllib.request, os

parser = PDBParser(QUIET=True)
print("Ready.")

## 1. Interface code, written properly this time

In week 1 you wrote the contact calculation inline, and it contained a bug: it counted waters and
sugars as antigen. Today we write it once, correctly, with the HETATM filter built in — and then
reuse it all session.

Note what the function signature forces you to state: which chains are antibody, which are antigen,
and what cutoff. Those were all implicit last time. Implicit parameters are where bugs live.

In [ ]:
def load(pdb_id):
    """Download (if needed) and parse a PDB entry, returning the first model."""
    path = f"{pdb_id}.pdb"
    if not os.path.exists(path):
        urllib.request.urlretrieve(f"https://files.rcsb.org/download/{pdb_id}.pdb", path)
    return parser.get_structure(pdb_id, path)[0]


def protein_atoms(model, chain_ids):
    """Heavy atoms of standard residues only - no waters, glycans, ions or hydrogens."""
    return [a for c in chain_ids for r in model[c] if r.id[0] == " "
            for a in r if a.element != "H"]


def interface(model, ab_chains, ag_chains, cutoff=4.5):
    """Compute the two sides of a protein-protein interface.

    Returns (paratope, epitope), each {(chain, resnum): (resname, atom_contacts)}.
    """
    ag_atoms = protein_atoms(model, ag_chains)
    search = NeighborSearch(ag_atoms)

    paratope, epitope = {}, Counter()
    epitope_names = {}

    for c in ab_chains:
        for r in model[c]:
            if r.id[0] != " ":
                continue
            n = 0
            for a in r:
                if a.element == "H":
                    continue
                for b in search.search(a.coord, cutoff):
                    n += 1
                    partner = b.get_parent()
                    key = (partner.get_parent().id, partner.id[1])
                    epitope[key] += 1
                    epitope_names[key] = partner.get_resname()
            if n:
                paratope[(c, r.id[1])] = (r.get_resname(), n)

    epitope = {k: (epitope_names[k], v) for k, v in epitope.items()}
    return paratope, epitope


# Sanity check against week 1's corrected answer.
m = load("1N8Z")
p, e = interface(m, ["A", "B"], ["C"])
print(f"trastuzumab paratope: {len(p)} residues   (week 1 corrected answer: 20)")
print(f"trastuzumab epitope:  {len(e)} residues")
assert len(p) == 20, "does not reproduce week 1"
print("\nReproduces week 1.")

## 2. Meet the second antibody

Pertuzumab's complex with HER2 is PDB `1S78`. Its chain layout is more complicated than `1N8Z`, and
working that out is your first task.

Rather than tell you the answer, let's print what is in the file.

In [ ]:
m2 = load("1S78")

print(f"{'chain':<8}{'residues':<11}{'numbered':<14}{'starts with'}")
print("-" * 62)
for ch in m2:
    res = [r for r in ch if r.id[0] == " "]
    if not res:
        continue
    seq = ""
    for r in res[:20]:
        try:
            seq += index_to_one(three_to_index(r.get_resname()))
        except Exception:
            seq += "X"
    print(f"{ch.id:<8}{len(res):<11}{f'{res[0].id[1]}-{res[-1].id[1]}':<14}{seq}")

### YOUR TURN — Question 1

Six chains. Work out what they are, using what you learned in weeks 1 and 2:

- Two are copies of the **antigen**. (Same trick as week 1 — they are much the longest.)
- Two are **light chains**, two are **heavy chains**. Light chains here are ~214 residues and begin
  `DIQM...`; heavy chains begin `EVQL...`.

Crystals often contain more than one copy of a complex in the asymmetric unit. Here there are **two
independent copies** of the same pertuzumab–HER2 complex. That is not a nuisance; it is a free
replicate, and we will use it.

One detail worth noticing now: look at the heavy chains' residue counts versus their numbering ranges.
They disagree. Why? (Week 2 has the answer.)

Fill in the chain assignment below.

In [ ]:
# YOUR TURN: which chains are which? Use the table above.
AG_1, AB_1 = "?", ["?", "?"]     # antigen and (light, heavy) for copy 1
AG_2, AB_2 = "?", ["?", "?"]     # antigen and (light, heavy) for copy 2

# --- self-check: a correct assignment produces a real interface, a wrong one produces nothing ---
try:
    p1, e1 = interface(m2, AB_1, [AG_1])
    p2, e2 = interface(m2, AB_2, [AG_2])
    if len(p1) > 10 and len(p2) > 10:
        print(f"Copy 1: paratope {len(p1)}, epitope {len(e1)}")
        print(f"Copy 2: paratope {len(p2)}, epitope {len(e2)}")
        print("\nBoth copies give a real interface. Correct.")
    else:
        print("These chains barely touch - you have paired an antibody with the wrong antigen copy.")
except Exception as err:
    print(f"Not valid yet ({type(err).__name__}). Chains are 'A'-'F'; "
          f"antigen is one chain, antibody is a [light, heavy] pair.")

## 3. The free replicate

Before comparing two *different* antibodies, use the two copies of the *same* one to ask a question
nobody asked in week 1:

> **If you measure the same interface twice, do you get the same answer?**

These are two copies of an identical complex, in the same crystal, solved in the same experiment. Any
difference between them is not biology. It is measurement noise — conformational wobble, differing
local resolution, crystal contacts, the crystallographer's modelling choices.

In [ ]:
# Compare by RESIDUE NUMBER, not by (chain, number): the two copies live in
# different chains, so the raw keys would never match even where the residue does.
set1 = {resnum for _, resnum in e1}
set2 = {resnum for _, resnum in e2}
shared = set1 & set2

print(f"copy 1 epitope: {len(set1)} residues")
print(f"copy 2 epitope: {len(set2)} residues")
print(f"shared:         {len(shared)}")
print(f"only in copy 1: {sorted(set1 - set2)}")
print(f"only in copy 2: {sorted(set2 - set1)}")

jaccard = len(shared) / len(set1 | set2)
print(f"\nJaccard overlap: {jaccard:.3f}")

### What that number means

The two copies agree almost perfectly — but not perfectly. A residue appears in one copy and not the
other, purely from measurement variation.

So **an interface is not a crisp set.** It has a well-agreed core and a fuzzy edge, and residues at
the edge flicker in and out depending on the copy, the cutoff, and the structure determination itself.

Keep this number in mind for the rest of the course. When you compare a *predicted* complex to a
crystal structure in week 6, you will want to know how much disagreement is meaningful. Here is a
lower bound on the noise floor, measured from a case where the right answer is "identical".

Anything smaller than this difference is not a finding.

## 4. The main event — do the two drugs compete?

Now the real comparison. Both antibodies bind HER2. The question is whether they bind the *same part*
of it.

This comparison is only possible because both PDB entries number HER2 the same way — same construct,
same residue numbering, starting at 1. **Check that before trusting any cross-structure comparison**;
if the numbering differed, you would have to align the sequences first, and comparing raw residue
numbers would give you nonsense that looks like a result.

In [ ]:
seq_1n8z = "".join(index_to_one(three_to_index(r.get_resname()))
                   for r in m["C"] if r.id[0] == " " and r.get_resname() != "UNK")[:60]
seq_1s78 = "".join(index_to_one(three_to_index(r.get_resname()))
                   for r in m2[AG_1] if r.id[0] == " " and r.get_resname() != "UNK")[:60]

print("1N8Z HER2 starts:", seq_1n8z)
print("1S78 HER2 starts:", seq_1s78)
print("\nSame construct and numbering?", seq_1n8z == seq_1s78)

In [ ]:
para_T, epi_T = interface(m,  ["A", "B"], ["C"])       # trastuzumab
para_P, epi_P = interface(m2, AB_1,       [AG_1])       # pertuzumab

res_T = sorted(r for _, r in epi_T)
res_P = sorted(r for _, r in epi_P)

print(f"trastuzumab epitope ({len(res_T)} residues):\n  {res_T}\n")
print(f"pertuzumab  epitope ({len(res_P)} residues):\n  {res_P}\n")

overlap = sorted(set(res_T) & set(res_P))
print(f"OVERLAP: {overlap if overlap else 'none - zero shared residues'}")
print(f"\ntrastuzumab spans HER2 residues {min(res_T)}-{max(res_T)}")
print(f"pertuzumab  spans HER2 residues {min(res_P)}-{max(res_P)}")

### The answer

**Zero overlap.** Not "small overlap" — the two epitopes do not share a single residue, and they sit
in completely different regions of the sequence: pertuzumab around residues 128–315, trastuzumab
around 557–605.

The HER2 extracellular region is built from four domains. Pertuzumab binds **domain II**, near the
surface HER2 uses to dimerise with other HER-family receptors. Trastuzumab binds **domain IV**, near
the membrane.

This is the structural basis for giving both drugs together: they do not compete, so a patient can be
dosed with both, and each blocks a different aspect of HER2 function. You just recovered a clinical
combination strategy from two coordinate files.

**Do not over-read it.** Non-overlapping epitopes make a combination *possible*; they do not make it
*work*. Whether a combination helps patients is a clinical question, settled by trials, not by a
distance calculation. What structure tells you is that the two antibodies *can* bind at once — a
necessary condition, not a sufficient one. Keep that distinction sharp; a great deal of
over-claiming in this field comes from treating a structural possibility as a functional result.

## 5. Contact maps

A residue list is hard to read. A **contact map** shows the interface as a grid — antibody residues
down one axis, antigen residues across the other, shaded by how many atom pairs are in contact.

Contact maps make interface *shape* visible: whether the binding is spread evenly or concentrated in
a few spots, and which loops carry it.

In [ ]:
def contact_matrix(model, ab_chains, ag_chains, cutoff=4.5):
    """Per-residue-pair contact counts, as a DataFrame (antibody rows x antigen cols)."""
    ag_atoms = protein_atoms(model, ag_chains)
    search = NeighborSearch(ag_atoms)
    pairs = Counter()
    for c in ab_chains:
        for r in model[c]:
            if r.id[0] != " ":
                continue
            for a in r:
                if a.element == "H":
                    continue
                for b in search.search(a.coord, cutoff):
                    pairs[(f"{c}{r.id[1]}", b.get_parent().id[1])] += 1
    if not pairs:
        return pd.DataFrame()
    rows = sorted({k[0] for k in pairs}, key=lambda s: (s[0], int(s[1:])))
    cols = sorted({k[1] for k in pairs})
    return pd.DataFrame([[pairs.get((r, c), 0) for c in cols] for r in rows],
                        index=rows, columns=cols)


fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
for ax, (title, mat) in zip(axes, [
        ("trastuzumab - HER2", contact_matrix(m,  ["A", "B"], ["C"])),
        ("pertuzumab - HER2",  contact_matrix(m2, AB_1,       [AG_1]))]):
    im = ax.imshow(mat.values, aspect="auto", cmap="viridis")
    ax.set_xticks(range(len(mat.columns)))
    ax.set_xticklabels(mat.columns, rotation=90, fontsize=7)
    ax.set_yticks(range(len(mat.index)))
    ax.set_yticklabels(mat.index, fontsize=7)
    ax.set_xlabel("HER2 residue")
    ax.set_ylabel("antibody residue")
    ax.set_title(title)
    fig.colorbar(im, ax=ax, label="atom contacts")
plt.tight_layout()
plt.show()

### YOUR TURN — Question 2

Compare the two maps:

1. Which antibody uses **more** residues to bind? Which buries a **larger** interface? (Compute BSA in
   the next section before answering the second part.)
2. Look at the row labels. For trastuzumab, both chains contribute about equally. What about
   pertuzumab — is it balanced, or dominated by one chain? Quote the split.
3. In week 1 you concluded that the "heavy chain dominates" folklore was wrong for trastuzumab.
   Does pertuzumab agree or disagree? What does having **two** data points do to a claim you made
   from **one**?
4. Are the contacts spread evenly, or concentrated in a few bright cells? What would a very
   concentrated interface imply about how easy it would be to disrupt with a single mutation?

*Double-click and write your answers here.*

1. Number of contacting residues: Count the distinct antibody residues that contact HER2. A residue should be counted only once, even if it makes several atomic contacts.
Buried surface area (BSA): We measure how much solvent-accessible surface area becomes buried when the antibody binds HER2. A larger BSA generally indicates a larger binding interface.
more contacting residues do not necessarily mean a larger BSA.
2. Here, \(N_L\) and \(N_H\) should be defined consistently. If we count residue pairs, a residue that contacts several residues on HER2 can contribute several contacts. If we count unique contacting residues, each residue contributes only once.
3.The scientific lesson is that one antibody is insufficient to establish a general rule about antibody interfaces.
If trastuzumab shows a nearly balanced contribution and pertuzumab shows a different split, the correct conclusion is that chain contributions depend on the particular antibody–antigen interaction.
Even if both antibodies show similar splits, two examples would provide supporting evidence, not proof that the pattern applies to all antibodies. Antibody binding often involves the complementarity-determining regions (CDRs) of both chains, but the relative contributions vary.
4.A highly concentrated interface could be more sensitive to a mutation that disrupts a key hotspot. However, a bright cell alone does not prove that a residue is energetically essential. A mutation can also alter folding, electrostatics, or the surrounding interaction network.

In [ ]:
sr = ShrakeRupley()


def sasa(model, chain_ids):
    """SASA of the given chains in isolation, protein atoms only."""
    import copy
    st = copy.deepcopy(model)
    for ch in list(st):
        if ch.id not in chain_ids:
            st.detach_child(ch.id)
    for ch in st:
        for r in [r for r in ch if r.id[0] != " "]:
            ch.detach_child(r.id)
    sr.compute(st, level="C")
    return sum(c.sasa for c in st)


def bsa_per_side(model, ab_chains, ag_chains):
    a = sasa(model, ab_chains)
    b = sasa(model, ag_chains)
    both = sasa(model, list(ab_chains) + list(ag_chains))
    return (a + b - both) / 2


rows = []
for name, mdl, ab, ag in [("trastuzumab", m, ["A", "B"], ["C"]),
                          ("pertuzumab",  m2, AB_1,      [AG_1])]:
    para, epi = interface(mdl, ab, ag)
    light = sum(1 for c, _ in para if c == ab[0])
    heavy = sum(1 for c, _ in para if c == ab[1])
    rows.append({
        "antibody": name,
        "paratope": len(para), "light": light, "heavy": heavy,
        "epitope": len(epi),
        "BSA_per_side": round(bsa_per_side(mdl, ab, ag)),
    })

pd.DataFrame(rows)

## 6. Hotspots — and what the word actually means

Which residues matter *most*? The obvious move is to rank by contact count and call the top few
"hotspots".

Let's do that, and then be honest about what we have produced.

In [ ]:
def top_contacts(paratope, n=6):
    ranked = sorted(paratope.items(), key=lambda kv: kv[1][1], reverse=True)
    return [(f"{c}{num}", resname, count) for (c, num), (resname, count) in ranked[:n]]


for name, para in [("trastuzumab", para_T), ("pertuzumab", para_P)]:
    print(f"{name} - most contacts:")
    for label, resname, count in top_contacts(para):
        print(f"   {label:<7}{resname:<6}{count} atom contacts")
    print()

aa_counts = Counter()
for para in (para_T, para_P):
    for (c, num), (resname, count) in para.items():
        aa_counts[resname] += 1
print("Amino acid composition of both paratopes combined:")
for aa, n in aa_counts.most_common(8):
    print(f"   {aa}  {n}")

### Two observations, one warning

**Observation 1 — tyrosine everywhere.** Look at the composition. Antibody paratopes are strikingly
enriched in aromatic residues, tyrosine above all. Tyrosine is big, flat, rigid, and can make both
hydrophobic and hydrogen-bonding contacts — a lot of binding capability for one residue, and cheap for
the immune system to encode. You will see this again in week 10, where language models trained on
antibody repertoires have clearly learned it, and in week 12, where designed CDRs come out
tyrosine-rich.

**Observation 2 — contacts concentrate.** A handful of residues make many contacts; most make few.

**The warning.** We have ranked residues by *contact count*, and then used the word *hotspot*. Those
are not the same thing.

A hotspot, as the term is actually used, is a residue whose mutation to alanine costs a large amount
of **binding free energy** — typically defined as ≥1–2 kcal/mol. That is a thermodynamic quantity.
It is measured experimentally by alanine scanning, and it correlates with contact count only loosely.
A residue can touch a lot of surface and contribute little; a buried residue making few contacts can
be worth several kcal/mol.

So what we have is a **contact ranking**, which is a reasonable place to *start* choosing mutations to
test. Calling it a hotspot list would be claiming knowledge of energetics we have not computed and
cannot compute from a single structure.

This distinction recurs all term. In week 6 you will meet models that output a confidence score, and
the temptation will be to read it as affinity. It is not. **Structure gives you geometry. Energy is a
separate question, and usually a harder one.**

## 7. Deliverable

Run the summary, then write the paragraph.

**Submit:** this notebook, executed top to bottom, committed as `week03/week03_comparing_interfaces.ipynb`.

Must contain: your chain assignment, the replicate-overlap number, the zero-overlap result, both
contact maps, the comparison table, and the paragraph.

In [ ]:
print("=" * 66)
print("WEEK 3 SUMMARY - two antibodies, one antigen")
print("=" * 66)
print(f"Replicate check (same complex, two crystal copies):")
print(f"  epitope Jaccard overlap      {jaccard:.3f}")
print(f"  residues differing            {len(set1 ^ set2)}")
print()
print(f"trastuzumab epitope  HER2 {min(res_T)}-{max(res_T)}  ({len(res_T)} residues)")
print(f"pertuzumab  epitope  HER2 {min(res_P)}-{max(res_P)}  ({len(res_P)} residues)")
print(f"shared residues:     {len(set(res_T) & set(res_P))}")
print()
print("=" * 66)

### Your paragraph

*Double-click and write 200–300 words.*

You are writing a short structural rationale for a colleague who wants to know **whether two
antibodies against the same target can be combined**. Use trastuzumab and pertuzumab as the worked
example, and your own numbers.

Cover:

- what you measured, and the cutoff and atom selection you used (be specific — week 1 taught you why)
- the epitope overlap result, and what it does and does not license you to claim
- the replicate comparison, and what it implies about how precisely an epitope boundary can be known
- one thing you would want that a crystal structure cannot give you

Write it as a technical note to a colleague, not an essay.

---

**Your answer:**Technical Note: Can Trastuzumab and Pertuzumab Be Used Together?**

 We compared the 3D structures of trastuzumab–HER2 (PDB: 1N8Z) and pertuzumab–HER2 (PDB: 1S78). We counted contacts between antibody and HER2 residues. A contact was defined as two non-hydrogen atoms being no more than 4.5 Å apart. We used this distance to identify which residues participate in binding.

 Trastuzumab binds to domain IV of HER2, whereas pertuzumab binds to domain II. Therefore, the antibodies recognize different regions of the same protein. This suggests that they may bind HER2 at the same time. However, different binding regions alone do not prove that simultaneous binding always occurs under real biological conditions.
 Small changes in the distance cutoff or the structure can change the number of contacting residues. For example, a residue at 4.4 Å is counted as a contact, whereas one at 4.6 Å is not. Therefore, the exact boundary of an epitope is not always clear. We should not report numerical variation unless we have actually repeated the calculations.

 A crystal structure shows the positions of atoms in a particular experimental structure, but it cannot tell us everything about how the antibodies behave in living cells. I would use surface plasmon resonance (SPR) to test whether both antibodies can bind HER2 simultaneously.**

## 9. Take-home exercises

Due before week 4.

**Exercise 1 — epitope robustness (required).**
Recompute both epitopes at 4.0, 4.5 and 5.0 Å. Does the zero-overlap conclusion survive all three
cutoffs? A conclusion that holds across a reasonable range of an arbitrary parameter is worth much
more than one that holds at a single value — this is the cheapest robustness check in structural
biology, and you should run it reflexively from now on.

An epitope here means the set of HER2 residues that contact the antibody. we calculate the overlap using the HER2 residue sets, not the antibody residues.
- If the intersection is empty at all three cutoffs, the zero-overlap result is robust to this particular parameter choice.
- If overlap appears at 5.0 Å, for example, the strict zero-overlap claim is cutoff-dependent.
- Either outcome concerns structural contacts; it does not, by itself, prove or disprove simultaneous binding in a biological system.

**Exercise 2 — paratope positions in a common frame (required).**
Using week 2's ANARCI code, convert both paratopes from PDB residue numbers into **IMGT** positions.
Then answer: do the two antibodies use the *same IMGT positions* to bind, even though they bind
completely different epitopes? This is the question raw residue numbers cannot answer and numbering
schemes exist to make askable.

The purpose of IMGT numbering is to put antibody residues into a common coordinate system. PDB residue 50 in one antibody is not necessarily equivalent to PDB residue 50 in another antibody.
The workflow is:
1. We identify the antibody heavy and light chains.
2. We find the antibody residues that contact HER2 at the chosen cutoff, such as 4.5 Å.
3. We extract each antibody chain’s sequence and map its PDB residue identifiers to the sequence positions.
4. Runing the Week 2 ANARCI workflow using the IMGT scheme.
5. Maping the contacting residues to their IMGT positions and compare the two antibodies’ sets.
Then we calculate:
- Shared IMGT positions: positions used by both antibodies.
- Heavy-chain paratope size: number of unique contacting heavy-chain residues.
- Light-chain paratope size: number of unique contacting light-chain residues.
The key point is that the antibodies could bind different parts of HER2 while using some of the same IMGT positions. Whether they actually do so must be established from the mapped residue sets; it cannot be concluded from the different epitopes alone.

**Exercise 3 — a third antibody (stretch).**
Pick any antibody–antigen complex from [SAbDab](https://opig.stats.ox.ac.uk/webapps/sabdab-sabpred/sabdab/)
and run today's pipeline on it end to end. Report paratope size, epitope size, BSA and chain split.
Does its heavy/light balance look like trastuzumab's, pertuzumab's, or neither?

For BSA, use the same calculation convention for all three complexes. For example, if we calculate the total loss of solvent-accessible surface area for both partners, we divide that total by two to report the buried area on one side of the interface. We say this convention because different tools and papers may report different BSA values.

In [ ]:
# Exercise 1 - cutoff robustness.
# You have `interface(model, ab_chains, ag_chains, cutoff=...)`. Loop over cutoffs.

# YOUR CODE HERE

In [ ]:
# Exercise 2 - map paratopes into IMGT positions.
#
# Hint: number the full chain with ANARCI, walk the occupied positions in order,
# and build {position_in_chain: imgt_label}. Week 2 section 7 does most of this.

def imgt_labels(model, chain_id):
    """Return {1-based position in chain: 'IMGT label'} for one antibody chain."""
    # YOUR CODE HERE
    pass

---

### Where we go next

Weeks 1–3 were all **measurement on solved structures**. You have been working with the answer key.

From week 4 the answer key goes away. You will *predict* structures and then ask how much to trust
them — and every comparison you make will lean on what you built today: a correct contact calculation,
a shared coordinate system, and a calibrated sense of how much difference is just noise.

| Week | Uses today's work |
|---|---|
| **4** | Predict Fv structures and measure error against crystal structures — using your RMSD/contact tooling. |
| **5** | Predict epitopes and compare to the *measured* epitopes you computed today. |
| **6** | Predict whole complexes. Your replicate-overlap number is the noise floor for judging a prediction. |
| **11** | Redesign a paratope — the one you now know how to define properly. |

Before week 4: finish exercises 1 and 2, and commit.